In [ ]:
import os, sys, shutil, subprocess, gc, json, time, traceback
from pathlib import Path
import numpy as np
import torch

WK  = Path("/kaggle/working"); WK.mkdir(parents=True, exist_ok=True)
INP = Path("/kaggle/input")

# Find dataset
SRC = INP / "datasets" / "krisskey" / "vericoding-urm"
if not SRC.exists():
    for r,_,fs in os.walk(str(INP)):
        if "submission_agent.py" in fs: SRC = Path(r); break
else:
    SRC = SRC

# Copy files
for fn in ["submission_agent.py","wasm_bridge.py","step_adapter.py",
           "frame_processor.py","game_profiler.py","dense_explorer.py","graph_explorer.py",
           "wasm_bridge.wasm","urm_checkpoint.pt","rhae_stage1.wasm"]:
    f = SRC / fn
    if f.exists(): shutil.copy2(f, WK / fn)
if (SRC/"external").is_dir():
    shutil.copytree(SRC/"external", WK/"external", dirs_exist_ok=True)
for d in [WK/"external", WK/"external"/"urm"]:
    d.mkdir(parents=True, exist_ok=True); (d/"__init__.py").touch()
for p in [str(WK), str(WK/"external")]:
    if p not in sys.path: sys.path.insert(0, p)
os.chdir(WK)

# Find environment files
for cand in [SRC/"environment_files", WK/"environment_files"]:
    if cand.is_dir() and list(cand.rglob("metadata.json")):
        os.environ["ENVIRONMENTS_DIR"] = str(cand); break
else:
    os.environ["ENVIRONMENTS_DIR"] = str(SRC/"environment_files")
print(f"OK: SRC={SRC.name}, ENV={os.environ['ENVIRONMENTS_DIR']}")

# Install arc-agi from dataset wheels
_whls = sorted(SRC.glob("*.whl"))
if _whls:
    for w in _whls:
        subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps",str(w)],
                       capture_output=True, timeout=30)
try:
    import arc_agi; print("arc-agi OK")
except Exception as e:
    print(f"arc-agi FAIL: {e}")


In [ ]:
import submission_agent as sa
from submission_agent import VERICODINGAgent
from step_adapter import safe_step
from wasm_bridge import functional_ttt_train, _extract_head_params, _extract_buffers
from collections import defaultdict

agent = VERICODINGAgent("__init__")
from wasm_bridge import _HAS_WASM
print(f"WASM: {'ACTIVE' if _HAS_WASM else 'Python fallback'}")

# URM backbone (CPU)
CHKPT = Path("/kaggle/working/urm_checkpoint.pt")
wm = None
if CHKPT.exists():
    wm = getattr(agent, "world_model", getattr(agent, "worldmodel", None))
    if wm is not None:
        load_fn = getattr(wm, "load_backbone", getattr(wm, "loadbackbone", None))
        if load_fn:
            import torch as _t
            _orig_cuda_avail = _t.cuda.is_available
            _t.cuda.is_available = lambda: False
            try:
                load_fn(str(CHKPT), device="cpu")
            finally:
                _t.cuda.is_available = _orig_cuda_avail
            wm.eval()
            print("URM backbone loaded (CPU)")
        else:
            wm = None
if hasattr(sa, '_carry_to_device'):
    _sa_orig = sa._carry_to_device
    def _safe_carry(carry, device):
        if not hasattr(carry, 'steps'): return carry
        return _sa_orig(carry, device)
    sa._carry_to_device = _safe_carry
    print(f"[patch] _carry_to_device OK")
else:
    print("[patch] _carry_to_device not in module - skipping")

print(f"Agent: {type(agent).__name__}, WM: {type(wm).__name__ if wm else None}")


In [ ]:
# ===== PHASE 0: DenseExplorer (plugged before agent loop) =====
# Runs 4-phase BFS/dense-scan per game. 10s timeout per game.
# If it finds a solution -> record score, skip agent loop for that game.
# Falls through to agent loop on failure (non-fatal).

from arc_agi import Arcade, EnvironmentInfo, OperationMode
from arcengine import GameState as _GS, enums as _GE
import submission_agent as _sa_mod

# ---- discover games ----
MAX_STEPS = 5000
DENSE_TIME_BUDGET = 10.0   # seconds per game for DenseExplorer

def _frame_win(nf):
    s = getattr(nf, "state", None) if nf is not None else None
    return s is not None and ("WIN" in str(s) or (hasattr(s,'value') and str(s.value)=="WIN"))

game_scores, trajectories = {}, {}
t0 = time.time()

_ENV_DIR = Path(os.environ.get("ENVIRONMENTS_DIR",""))
env_infos = []
for mf in sorted(_ENV_DIR.rglob("metadata.json")):
    jd = json.loads(mf.read_text())
    jd["local_dir"] = str(mf.parent)
    env_infos.append(EnvironmentInfo.model_validate(jd))
print(f"Games: {len(env_infos)}")

arc = Arcade(operation_mode=OperationMode.OFFLINE)

# ---- build canonical GameAction list once ----
_GA = _GE.GameAction
_ga_all = list(_GA)    # [ACTION1..ACTION7]
print(f"GameActions: {[a.name for a in _ga_all]}")

# ---- DenseExplorer bootstrap ----
try:
    from dense_explorer import DenseExplorer
    _dense_available = True
    print("DenseExplorer: LOADED")
except Exception as _de_err:
    _dense_available = False
    print(f"DenseExplorer: UNAVAILABLE ({_de_err})")

dense_wins = 0
dense_skipped = 0

for idx, ei in enumerate(env_infos):
    gid = str(getattr(ei, "game_id", getattr(ei, "id", idx)))
    env = None; score = 0.0
    env = arc.make(gid)

    # =========================================================
    # PHASE 0: DenseExplorer                                   
    # env.reset() returns a single Frame (NOT a tuple).        
    # DenseExplorer uses safe_step internally (handles complex).
    # =========================================================
    dense_solved = False
    if _dense_available:
        try:
            _t_dense = time.time()
            # Build action_list = GameAction objects (have .is_complex())
            _act_list_dense = list(getattr(env, "action_space", _ga_all))
            if _act_list_dense and not hasattr(_act_list_dense[0], "is_complex"):
                _act_list_dense = _ga_all  # fallback to canonical
            if _act_list_dense:
                env.reset()   # Frame returned but DenseExplorer resets itself
                explorer = DenseExplorer(env, _act_list_dense)
                # Cap steps so we stay within time budget
                # ~200 steps/s => 2000 steps in 10s
                found = explorer.explore(max_steps=2000)
                _elapsed_dense = time.time() - _t_dense
                if found and explorer.solution:
                    # Replay solution: each item is int (action idx) or (aidx,cx,cy) tuple
                    env.reset()
                    _rscore = 0.0
                    _won = False
                    for _item in explorer.solution:
                        if isinstance(_item, (list, tuple)) and len(_item) == 3:
                            _aidx, _cx, _cy = int(_item[0]), int(_item[1]), int(_item[2])
                        else:
                            _aidx, _cx, _cy = int(_item), 32, 32
                        if _aidx >= len(_act_list_dense): break
                        _aobj = _act_list_dense[_aidx]
                        _nf = safe_step(env, _aobj, _cx, _cy)
                        if _nf is None: break
                        _pframe_lvl = getattr(env.reset(), "levels_completed", 0)
                        _rscore += float(getattr(_nf, "levels_completed", 0) - 0)
                        if _frame_win(_nf):
                            _won = True; break
                    if _won or _rscore > 0:
                        game_scores[gid] = _rscore
                        dense_solved = True
                        dense_wins += 1
                        print(f"  [{idx+1}] {gid}: DENSE WIN score={_rscore:.4f} t={_elapsed_dense:.1f}s")
                    else:
                        print(f"  [{idx+1}] {gid}: dense no-score ({explorer._total_steps} steps, {_elapsed_dense:.1f}s)")
                else:
                    print(f"  [{idx+1}] {gid}: dense no-win ({getattr(explorer,'_total_steps',0)} steps, {time.time()-_t_dense:.1f}s)")
            else:
                dense_skipped += 1
        except Exception as _de:
            print(f"  [{idx+1}] {gid}: dense ERR (non-fatal): {str(_de)[:120]}")
            print(traceback.format_exc()[:400])

    if dense_solved:
        try: env.close()
        except: pass
        gc.collect()
        continue   # skip agent loop for this game

    # =========================================================
    # PHASE 1: Agent loop (original pipeline)                  
    # =========================================================
    _frames = [None]; nf = None; sbuf, abuf = [], []
    try:
        agent.set_game_tags(getattr(getattr(env,"info",None),"tags",[]))
        agent.set_step_modulus(3)
        _frames = [env.reset()]
        if _frames[0] is None:
            game_scores[gid] = 0.0
            continue
        agent.on_game_start()

        _sa_mod = __import__("submission_agent")
        if hasattr(_sa_mod, "_rhae"):
            _rhae = _sa_mod._rhae
            if _rhae._exp is None:
                _rhae._exp = defaultdict(lambda: lambda *a,**kw: None)

        for _ in range(MAX_STEPS):
            act = agent.choose_action(_frames, None)
            if hasattr(act, 'action') and not hasattr(act, 'is_complex'):
                _aidx = act.action
                _game_act = _ga_all[_aidx % len(_ga_all)]
                ad = None
            else:
                _game_act = act
                ad = getattr(agent, "_last_action_data", None)
            pframe = _frames[-1]
            if wm is not None and pframe is not None:
                grid = getattr(pframe,"frame",None)
                if grid is not None and len(grid)>0:
                    g = np.asarray(grid[0], dtype=np.int32)
                    _aid = _aidx if '_aidx' in dir() else 0
                    tok = wm.encode_state(g, _aid)
                    sbuf.append(tok.squeeze(0).cpu().numpy().astype(np.int32))
                    abuf.append(_aid)
            if ad:
                nf = safe_step(env, _game_act, x=ad.get("x"), y=ad.get("y"))
            else:
                nf = safe_step(env, _game_act)
            if nf is None: break
            score += float(getattr(nf,"levels_completed",0)-getattr(pframe,"levels_completed",0))
            _frames.append(nf)
            if getattr(nf,"state",None) is _GS.GAME_OVER: break
            if _frame_win(nf): break

        game_scores[gid] = score
        print(f"  [{idx+1}] {gid}: score={score:.4f}, steps={max(0,len(_frames)-1)}")

        # Per-game TTT
        if len(sbuf)>=4 and wm is not None:
            try:
                tdev = "cpu"
                if torch.cuda.is_available():
                    try:
                        cap = torch.cuda.get_device_capability(0)
                        if cap[0] >= 7: tdev = "cuda"
                    except: pass
                s = torch.from_numpy(np.stack(sbuf)).long().to(tdev)
                a = torch.from_numpy(np.array(abuf)).long().to(tdev)
                ns = torch.cat([s[1:],s[-1:]],dim=0)
                rw = torch.zeros(len(s), device=tdev)
                params = {k:v.to(tdev).requires_grad_(True) for k,v in _extract_head_params(wm).items()}
                bufs = {k:v.to(tdev) for k,v in _extract_buffers(wm).items()}
                new_p = functional_ttt_train(params, bufs, wm, s, a, ns, rw, steps=30, lr=8e-5, lambda_reg=0.1)
                for k,v in wm.named_parameters():
                    if k in new_p: v.data.copy_(new_p[k].cpu())
                print(f"    [TTT] params applied (tdev={tdev}, score={score:.2f})")
                if tdev == "cuda": torch.cuda.empty_cache()
            except Exception as _tte:
                print(f"    [TTT] skipped: {_tte}")

    except Exception as e:
        game_scores[gid] = 0.0
        print(f"  [{idx+1}] {gid} ERR: {str(e)[:100]}")
    finally:
        if env:
            try: env.close()
            except: pass
        gc.collect()

mean_s = float(np.mean(list(game_scores.values()))) if game_scores else 0.0
n_traj = len(game_scores)
print(f"\nPhase A done: {n_traj}/{len(env_infos)} games, mean_score={mean_s:.4f}, t={time.time()-t0:.0f}s")
print(f"DenseExplorer wins: {dense_wins}/{len(env_infos)}")


In [ ]:
ADAPTER = Path("/kaggle/working/urm_ttt_adapter.pt")
METRICS = Path("/kaggle/working/training_metrics.json")

if wm is not None:
    ad = _extract_head_params(wm)
    ad = {k:v for k,v in ad.items() if "action_emb" not in k}
    torch.save(ad, ADAPTER)
else:
    torch.save({}, ADAPTER)

metrics = {"n_games":len(env_infos),"n_trajectories":len(trajectories),
           "dense_wins":dense_wins,
           "pre_ttt_mean":float(np.mean(list(game_scores.values()))) if game_scores else 0.0,
           "ts":time.strftime("%Y-%m-%dT%H:%M:%S")}
METRICS.write_text(json.dumps(metrics, indent=2))
print(f"=== DONE === mean={metrics['pre_ttt_mean']:.4f} | dense_wins={dense_wins}")
